# Milo voice — g7 in Josh

Clones branch `explain-g7` and renders every Josh line of **g7** that has no clip yet — **932 lines** (a few fewer if lines shared with another module were rendered first).
Styles: B (0.5/0.5) and B+ (0.7/0.3) with the original Chatterbox, A with Turbo. Rules: `docs/new-flow/voice.md`.

**Before Run All:** Settings → Accelerator → **GPU T4 x2** (or P100), Internet **On**. At the end, download
`milo-voice-josh-g7.zip` from the Output panel and hand it back.

In [ ]:
import os, sys, subprocess, pathlib, shutil, json
BRANCH = 'explain-g7'
MODULE = 'g7'
PREFIX = 'g7' + ('m' if 'm' not in 'g7' else '-')
SKIP = []
VOICE = 'nzFihrBIvB34imQBuxub'   # Josh
WORK = pathlib.Path('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
REPO = WORK / 'learn'
# A Kaggle session outlives a notebook: a copy cloned by ANOTHER grade's notebook would be read as this one's (0 lines).
if REPO.exists() and subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip() != BRANCH:
    shutil.rmtree(REPO)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/RadlorInc/learn.git', str(REPO)], check=True)
rows = json.load(open(REPO / 'scripts/.voice-corpus-lessons-josh.json'))
todo = [r for r in rows if any(s.startswith(PREFIX) and not any(s.startswith(k + '-') for k in SKIP) for s in r['sources'])
        and not (REPO / 'public/audio' / VOICE / f"{r['key']}.mp3").exists()]
json.dump(todo, open(WORK / 'todo.json', 'w'))
print(len(todo), 'lines to render ·', {s: sum(r['style'] == s for r in todo) for s in ['A', 'B', 'B+']})
# At most 932: lines shared with another module ("Okay. Your turn.") may already be rendered by the time this runs.
assert 0 < len(todo) <= 932, f'expected up to 932 lines for g7 — is the branch right?'

In [ ]:
# Chatterbox pins torch 2.6, which breaks Kaggle's own torchvision — so it gets its own venv (uv: Kaggle's python has no ensurepip).
VENV = WORK / 'venv'; PY = VENV / 'bin' / 'python'; READY = VENV / '.ready'
if not READY.exists():
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', str(VENV), '--python', sys.executable], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PY), 'setuptools<81', 'chatterbox-tts', 'imageio-ffmpeg'], check=True)
    READY.touch()
print(subprocess.run([str(PY), '-c', "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set the accelerator')"], capture_output=True, text=True).stdout)

In [ ]:
OUT = WORK / 'out' / VOICE
cmd = [str(PY), str(REPO / 'scripts/chatterbox-render.py'), '--voice', VOICE, '--corpus', str(WORK / 'todo.json'), '--out', str(OUT)]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if 'it/s]' in line or not line.strip(): continue   # drop the progress bars
    print(line.rstrip()[:160], flush=True)
print('exit code', proc.wait())

In [ ]:
clips = sorted(OUT.glob('*.mp3'))
missing = [r['key'] for r in todo if not (OUT / f"{r['key']}.mp3").exists()]
stage = WORK / 'stage'; shutil.rmtree(stage, ignore_errors=True); (stage / VOICE).mkdir(parents=True)
for c in clips: shutil.copy2(c, stage / VOICE / c.name)
shutil.make_archive(str(WORK / 'milo-voice-josh-g7'), 'zip', stage)
print(f'{len(clips)} of {len(todo)} clips in milo-voice-josh-g7.zip' + (f' — MISSING {len(missing)}: {missing}' if missing else ' — complete'))